# Section 2 · Notebook 3 — When Simple Tools Aren't Enough (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

1. The database connection problem — and the class-based fix
2. Slow APIs — async tools that run in parallel

We use a small local SQLite database so you can count connections for real.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

In [ ]:
import sqlite3

DB_PATH = "inventory_demo.db"

# Create and seed a small inventory database
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("DROP TABLE IF EXISTS inventory")
    conn.execute("CREATE TABLE inventory (id TEXT PRIMARY KEY, name TEXT, quantity INTEGER, price REAL)")
    conn.executemany(
        "INSERT INTO inventory VALUES (?, ?, ?, ?)",
        [
            ("PROD-123", "Wireless Mouse", 15, 29.99),
            ("PROD-456", "USB-C Hub", 0, 49.99),
            ("PROD-789", "Mechanical Keyboard", 8, 89.99),
        ],
    )
print("Database ready:", DB_PATH)

CONNECTIONS_OPENED = 0


def connect_to_database():
    """Open a new database connection and count it."""
    global CONNECTIONS_OPENED
    CONNECTIONS_OPENED += 1
    # check_same_thread=False: the agent may run tools on worker threads
    return sqlite3.connect(DB_PATH, check_same_thread=False)

---
## 1. The Database Connection Problem

Each tool below opens its own connection, uses it once, and closes it. With a handful of calls that looks harmless. In production, with many users and many tool calls per request, it means hundreds of connections a minute — and an unhappy database administrator.

In [ ]:
from strands import Agent, tool


@tool
def check_stock(product_id: str) -> str:
    """Check product stock."""
    db = connect_to_database()
    row = db.execute("SELECT quantity FROM inventory WHERE id = ?", (product_id,)).fetchone()
    db.close()
    return f"{product_id}: {row[0]} units" if row else f"{product_id} not found"


@tool
def update_stock(product_id: str, quantity: int) -> str:
    """Update product stock."""
    db = connect_to_database()
    db.execute("UPDATE inventory SET quantity = ? WHERE id = ?", (quantity, product_id))
    db.commit()
    db.close()
    return f"Updated {product_id} to {quantity} units"


CONNECTIONS_OPENED = 0
agent = Agent(model=model, tools=[check_stock, update_stock])
agent("Check stock for PROD-123 and PROD-789, set PROD-456 to 25 units, then confirm PROD-456's new level")

print(f"\n\nConnections opened for one request: {CONNECTIONS_OPENED}")

### The fix: group related tools in a class

Connect once in `__init__`, share the connection through `self`. Every tool method still uses `@tool`. Because the agent may call several tools at the same time, the shared connection is protected with a lock.

Always use parameterised queries (`?` placeholders) rather than f-strings inside SQL — the values come from the model, and you should treat them like user input.

In [ ]:
import threading


class InventoryTools:
    def __init__(self):
        # One shared connection for every tool on this object.
        # The agent can run tools in parallel threads, so guard the shared
        # connection with a lock.
        self.db = connect_to_database()
        self.lock = threading.Lock()

    @tool
    def check_stock(self, product_id: str) -> str:
        """Check product stock level.

        Args:
            product_id: The product ID to check
        """
        with self.lock:
            row = self.db.execute("SELECT name, quantity FROM inventory WHERE id = ?", (product_id,)).fetchone()
        return f"{row[0]} ({product_id}): {row[1]} units in stock" if row else f"Product {product_id} not found"

    @tool
    def update_stock(self, product_id: str, quantity: int) -> str:
        """Update product stock quantity.

        Args:
            product_id: The product ID to update
            quantity: New quantity to set
        """
        with self.lock:
            cur = self.db.execute("UPDATE inventory SET quantity = ? WHERE id = ?", (quantity, product_id))
            self.db.commit()
        return f"Updated {product_id} to {quantity} units" if cur.rowcount else f"Product {product_id} not found"


CONNECTIONS_OPENED = 0
inventory = InventoryTools()
agent = Agent(model=model, tools=[inventory.check_stock, inventory.update_stock])
agent("Check stock for PROD-123 and PROD-789, set PROD-456 to 40 units, then confirm PROD-456's new level")

print(f"\n\nConnections opened for one request: {CONNECTIONS_OPENED}")

---
## 2. Slow Warehouse Checks — Async Tools

Imagine checking three warehouses, each API taking 2 seconds. Called one after another:

```python
warehouse_a = check_warehouse_inventory("PROD-123", "warehouse_a")  # 2 seconds
warehouse_b = check_warehouse_inventory("PROD-123", "warehouse_b")  # 2 seconds
warehouse_c = check_warehouse_inventory("PROD-123", "warehouse_c")  # 2 seconds
# Total: 6 seconds of waiting
```

Make the tool `async` and use `await` inside it. When the model asks for all three checks in one turn, Strands runs them concurrently — about 2 seconds in total. Call the agent with `invoke_async`.

> **Groq note:** `gpt-oss` models on Groq request one tool per turn, so the three checks would still run one after another. This demo uses `qwen/qwen3.8-27b`, which asks for all three checks in a single turn — that is what lets Strands run them concurrently.

In [ ]:
import asyncio
import time


@tool
async def check_warehouse_inventory(product_id: str, warehouse: str) -> dict:
    """Check inventory at a specific warehouse.

    Args:
        product_id: Product ID to check
        warehouse: Warehouse identifier (e.g., "east", "west", "central")
    """
    await asyncio.sleep(2)  # simulate a slow API call

    data = {
        "east": {"PROD-123": 45, "PROD-456": 12},
        "west": {"PROD-123": 30, "PROD-456": 0},
        "central": {"PROD-123": 60, "PROD-456": 25},
    }
    quantity = data.get(warehouse, {}).get(product_id, 0)
    return {"warehouse": warehouse, "product_id": product_id, "quantity": quantity}


# A model that asks for several tool calls in one turn (see the note above)
PARALLEL_MODEL_ID = os.getenv("GROQ_PARALLEL_MODEL_ID", "qwen/qwen3.8-27b")
parallel_model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=PARALLEL_MODEL_ID,
    params={"extra_body": {"include_reasoning": False}},
)


async def main():
    agent = Agent(model=parallel_model, tools=[check_warehouse_inventory])
    start = time.time()
    response = await agent.invoke_async(
        "Can we ship 100 units of PROD-123? Check all warehouses: east, west, and central."
    )
    elapsed = time.time() - start
    print("\n\n" + response.message["content"][0]["text"])
    print(f"\nTotal time: {elapsed:.1f}s (three sequential checks alone would take ~6s)")


await main()

In a real tool you would use an async HTTP client, for example:

```python
import httpx

@tool
async def check_warehouse_inventory(product_id: str, warehouse: str) -> int:
    """Check inventory at a specific warehouse."""
    async with httpx.AsyncClient() as client:
        response = await client.get(f"https://api.warehouse-{warehouse}.example.com/inventory/{product_id}")
    return response.json()["quantity"]
```